# SkyEye — 天气分类
## EfficientNet-B5 → 知识蒸馏 → B0 → 结构化剪枝 → ONNX 导出

按顺序执行以下 Cell：

| Cell | 内容 |
|------|------|
| 1 | 安装依赖 |
| 2-4 | 数据准备（Mo 平台 !cp -R / !7zx） |
| 5-6 | 环境检查 + DataLoader |
| 7 | 训练教师模型 (EfficientNet-B5) |
| 8 | 知识蒸馏 (B5 → B0) |
| 9 | 结构化剪枝 + 微调 |
| 10 | ONNX 导出 + INT8 量化 + 测速 |
| 11 | (可选) CPU 单张推理测试 |

In [ ]:
# Cell 1: 安装依赖（云端 PyTorch / torchvision 已预装，跳过）
!pip install timm==1.0.8 onnx==1.16.1 onnxruntime-gpu==1.18.1 tqdm numpy==1.26.4 scipy==1.12.0 scikit-learn==1.5.2

In [ ]:
# Cell 2: 探索可用的数据集导入
import os

datasets_root = "/home/jovyan/work/datasets"
print("=" * 60)
print("已导入的数据集:")
print("=" * 60)

# 记录找到的数据源路径
weather_src = None  # weather_classification 目录
zip_src = None      # data_split.zip 文件

for import_name in sorted(os.listdir(datasets_root)):
    import_dir = os.path.join(datasets_root, import_name)
    if not os.path.isdir(import_dir) or import_name.startswith('.'):
        continue
    print(f"\\n[{import_name}]")
    for item in sorted(os.listdir(import_dir)):
        item_path = os.path.join(import_dir, item)
        if os.path.isdir(item_path) and item == "weather_classification":
            print(f"  {item}/  ← 天气分类数据集")
            weather_src = item_path
            # 列出类别
            for cls in sorted(os.listdir(item_path)):
                cls_path = os.path.join(item_path, cls)
                if os.path.isdir(cls_path):
                    count = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
                    print(f"    {cls}/ ({count} 张)")
        elif item.lower().endswith('.zip'):
            size_mb = os.path.getsize(item_path) / (1024 * 1024)
            print(f"  {item} ({size_mb:.1f} MB) ← 压缩数据集")
            zip_src = item_path
        else:
            print(f"  {item}/")

print(f"\\n{'='*60}")
print(f"weather_classification: {weather_src}")
print(f"data_split.zip: {zip_src}")
print(f"{'='*60}")

In [ ]:
# Cell 3: 复制 weather_classification 到可写目录（Mo 平台标准做法）
# 注意：datasets/ 是只读的，必须用 !cp -R 复制出来才能使用
import os

# 使用 Cell 2 中自动发现的路径
if weather_src and not os.path.exists("_data/weather_src1"):
    print(f"正在复制: {weather_src} → _data/weather_src1")
    !cp -R {weather_src} _data/weather_src1
    print("复制完成")
elif os.path.exists("_data/weather_src1"):
    print("_data/weather_src1 已存在，跳过复制")
else:
    print("⚠ 未找到 weather_classification，请确认数据集已导入")

In [ ]:
# Cell 4: 解压 data_split.zip 到可写目录（Mo 平台标准做法）
import os

if zip_src and not os.path.exists("_data/weather_src2"):
    print(f"正在解压: {zip_src} → _data/weather_src2")
    os.makedirs("_data/weather_src2", exist_ok=True)
    !7zx {zip_src} -d _data/weather_src2
    print("解压完成")
elif os.path.exists("_data/weather_src2"):
    print("_data/weather_src2 已存在，跳过解压")
else:
    print("⚠ 未找到 data_split.zip，将仅使用 weather_classification")

In [ ]:
# Cell 5: 数据合并 — 自动类名映射 + 多源合并到 _data/weather
import os
import sys

from config import CONFIG

# 构建显式数据源列表（从 Cell 3-4 准备好的可写副本）
data_sources = []
if os.path.exists("_data/weather_src1"):
    data_sources.append({
        "path": "_data/weather_src1",
        "class_map": {"haze": "foggy", "snow": "snowy", "thunder": "thundery"}
    })
if os.path.exists("_data/weather_src2"):
    data_sources.append({
        "path": "_data/weather_src2",
        "class_map": {}  # zip 内类名已是标准形式
    })

if data_sources:
    # 临时覆盖 CONFIG，使用已复制的数据源
    CONFIG["data_roots"] = data_sources
    print(f"数据源: {[s['path'] for s in data_sources]}")
    
    from data.dataset import prepare_data
    data_root = prepare_data()
    print(f"\\n数据已就绪: {data_root}")
else:
    print("❌ 没有可用的数据源！请先运行 Cell 3-4 并确认数据集已导入")

In [ ]:
# Cell 6: 环境检查 + 创建 DataLoader
from config import CONFIG
import torch

print(f'Device: {CONFIG["device"]}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Classes: {CONFIG["class_names"]}')
print(f'Teacher: {CONFIG["teacher_model"]}, Student: {CONFIG["student_model"]}')

from data.dataset import create_dataloaders
train_loader, val_loader, class_counts = create_dataloaders()
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Cell 7: 训练教师模型 (EfficientNet-B5)
from training.train_teacher import train_teacher
teacher = train_teacher()

In [ ]:
# Cell 8: 知识蒸馏 (B5 → B0)
from training.distill_student import run_distillation
student = run_distillation()

In [ ]:
# Cell 9: 结构化剪枝 + 微调
from training.prune_finetune import prune_and_finetune
pruned_model = prune_and_finetune()

In [ ]:
# Cell 10: ONNX 导出 + INT8 量化 + CPU 推理测速
from inference.export_onnx import export_to_onnx, quantize_to_int8, benchmark_cpu
onnx_path = export_to_onnx('results/student_pruned_final.pth')
int8_path = quantize_to_int8(onnx_path)
benchmark_cpu(onnx_path, int8_path)

In [ ]:
# Cell 11 (可选): CPU 单张推理测试
from inference.infer import predict_image
result = predict_image('test_image.jpg')  # 替换为实际图片路径
print(f'Prediction: {result["prediction"]} (Confidence: {result["confidence"]:.4f})')
for name, prob in result['top_k']:
    print(f'  {name}: {prob:.4f}')